# Ethene grid generator

From the initial ethene grid, choose geometries `STRIDE` points apart.

Then find the normal modes of ethene.
For each geometry, displace it in the direction of each mode in `SELECTED_MODES`.

Thus get a grid of dimension `2 + len(SELECTED_MODES)`.

In [8]:
from pathlib import Path

import numpy as np

SOURCE_XYZ = Path("../data/A01_ethene/static/A01_ethene_grid_static_CASSCF_FINAL.xyz")
OUTPUT_XYZ = Path("../data/A01_ethene/static/A01_ethene_grid_static_CASSCF_every3.xyz")
STRIDE = 3


def h_c_c_h_dihedral(positions):
    """Return the H(2)-C(0)-C(1)-H(3) dihedral in degrees."""
    a, b, c, d = positions[2], positions[0], positions[1], positions[3]
    bond0 = -(b - a)
    bond1 = c - b
    bond2 = d - c
    bond1 /= np.linalg.norm(bond1)
    v = bond0 - np.dot(bond0, bond1) * bond1
    w = bond2 - np.dot(bond2, bond1) * bond1
    return np.degrees(np.arctan2(np.dot(np.cross(bond1, v), w), np.dot(v, w)))


def read_raw_xyz_frames(path):
    """Read frames while retaining their original raw extended-XYZ text."""
    lines = path.read_text().splitlines(keepends=True)
    frames = []
    cursor = 0
    while cursor < len(lines):
        if not lines[cursor].strip():
            cursor += 1
            continue
        natoms = int(lines[cursor])
        stop = cursor + natoms + 2
        if stop > len(lines):
            raise ValueError(f"Incomplete frame starting at line {cursor + 1}")
        atom_lines = lines[cursor + 2 : stop]
        positions = np.array([[float(value) for value in line.split()[1:4]] for line in atom_lines])
        if positions.shape != (6, 3):
            raise ValueError(f"Expected six atoms, got {positions.shape[0]}")
        frames.append({"source_frame_index": len(frames), "raw": "".join(lines[cursor:stop]), "positions": positions})
        cursor = stop
    return frames


frames = read_raw_xyz_frames(SOURCE_XYZ)
for frame in frames:
    positions = frame["positions"]
    frame["bond"] = round(float(np.linalg.norm(positions[1] - positions[0])), 6)
    frame["dihedral"] = round(float(h_c_c_h_dihedral(positions)), 6)

bond_coordinates = sorted({frame["bond"] for frame in frames})
dihedral_coordinates = sorted({frame["dihedral"] for frame in frames})
if (len(bond_coordinates), len(dihedral_coordinates)) != (41, 91):
    raise ValueError(
        "Unexpected grid dimensions: "
        f"{len(bond_coordinates)} bond coordinates x {len(dihedral_coordinates)} dihedrals"
    )

frame_by_coordinate = {}
for frame in frames:
    key = (frame["bond"], frame["dihedral"])
    if key in frame_by_coordinate:
        raise ValueError(f"Duplicate grid coordinate: {key}")
    frame_by_coordinate[key] = frame

expected_grid_size = len(bond_coordinates) * len(dihedral_coordinates)
if len(frame_by_coordinate) != expected_grid_size:
    raise ValueError(
        f"Incomplete grid: found {len(frame_by_coordinate)} of {expected_grid_size} coordinates"
    )

selected_bonds = bond_coordinates[::STRIDE]
selected_dihedrals = dihedral_coordinates[::STRIDE]
selected_keys = [(bond, dihedral) for bond in selected_bonds for dihedral in selected_dihedrals]
missing_keys = [key for key in selected_keys if key not in frame_by_coordinate]
if missing_keys:
    raise ValueError(f"Missing selected grid coordinates: {missing_keys[:5]}")
if len(selected_keys) != 14 * 31:
    raise ValueError(f"Expected 434 selected geometries, got {len(selected_keys)}")

OUTPUT_XYZ.write_text("".join(frame_by_coordinate[key]["raw"] for key in selected_keys))

print(f"Wrote {len(selected_keys)} geometries to {OUTPUT_XYZ.resolve()}")
print(f"Grid shape: {len(selected_bonds)} bond coordinates x {len(selected_dihedrals)} dihedrals")


Wrote 434 geometries to /home/lim_yt/X-MACE-sampling/data/A01_ethene/static/A01_ethene_grid_static_CASSCF_every3.xyz
Grid shape: 14 bond coordinates x 31 dihedrals


In [9]:
from ase import Atoms
from xtb.ase.calculator import XTB
from wfl.configset import ConfigSet, OutputSpec
from wfl.generate import normal_modes as nm

REFERENCE_NM_XYZ = Path("ethene_reference_normal_modes.xyz")
reference_bond = min(bond_coordinates, key=lambda value: abs(value - 1.33))
reference_dihedral = min(dihedral_coordinates, key=abs)
reference_frame = frame_by_coordinate[(reference_bond, reference_dihedral)]
reference_atoms = Atoms(
    symbols=["C", "C", "H", "H", "H", "H"],
    positions=reference_frame["positions"],
    pbc=False,
)
configset = ConfigSet([reference_atoms])
outputspec = OutputSpec(REFERENCE_NM_XYZ, overwrite=True)

calc = (XTB, [], {"method": "GFN2-xTB"})
prop_prefix = "xtb2_"

nm.generate_normal_modes_parallel_hessian(inputs=configset,
                                          outputs=outputspec,
                                          calculator=calc,
                                          prop_prefix=prop_prefix)

print(
    f"Wrote normal modes for C-C = {reference_bond:.6f} Å, "
    f"dihedral = {reference_dihedral:.1f}° to {REFERENCE_NM_XYZ.resolve()}"
)

Wrote normal modes for C-C = 1.336172 Å, dihedral = 0.0° to /home/lim_yt/X-MACE-sampling/notebooks/ethene_reference_normal_modes.xyz


In [10]:
from ase.io import read
from wfl.generate import normal_modes as nm

at = read(REFERENCE_NM_XYZ)
ethene_nm = nm.NormalModes(at, "xtb2_")

# Writes trajectories of the selected normal modes to file.
ethene_nm.view()

# prints frequencies
ethene_nm.summary()

---------------------

  #    meV     cm^-1

---------------------

  0   18.9 i   152.1 i
  1    0.0 i     0.0 i
  2    0.0 i     0.0 i
  3   48.1     387.7  
  4   56.0     451.8  
  5   60.2     485.5  
  6  100.5     810.4  
  7  107.0     863.2  
  8  117.7     949.4  
  9  125.7    1013.6  
 10  131.6    1061.2  
 11  170.6    1375.9  
 12  175.4    1414.4  
 13  201.8    1627.4  
 14  376.7    3038.2  
 15  377.0    3040.5  
 16  381.7    3078.9  
 17  382.6    3086.2  
---------------------



In [ ]:
from itertools import product

from ase import Atoms
from ase.io import read, write

AUGMENTED_OUTPUT_XYZ = Path(
    "../data/A01_ethene/static/A01_ethene_grid_static_CASSCF_extended.xyz" 
)
SELECTED_MODES = (14, 15, 16, 17)
NORMAL_COORDINATES = (-0.10, 0.10)  # Å multipliers of normalized mode vectors

normal_mode_atoms = read(REFERENCE_NM_XYZ)
ethene_nm = nm.NormalModes(normal_mode_atoms, prop_prefix)
if max(SELECTED_MODES) >= ethene_nm.num_nm:
    raise ValueError(f"Selected modes {SELECTED_MODES} are unavailable")
mode_vectors = {mode: ethene_nm.modes[mode] for mode in SELECTED_MODES}

augmented_atoms = []
for base_key in selected_keys:
    base_frame = frame_by_coordinate[base_key]
    for mode_coordinates in product(NORMAL_COORDINATES, repeat=len(SELECTED_MODES)):
        displacement = sum(
            coordinate * mode_vectors[mode]
            for mode, coordinate in zip(SELECTED_MODES, mode_coordinates)
        )
        atoms = Atoms(
            symbols=["C", "C", "H", "H", "H", "H"],
            positions=base_frame["positions"] + displacement,
            pbc=False,
        )
        atoms.info = {
            "source_frame_index": base_frame["source_frame_index"],
            "grid_cc_bond_angstrom": base_key[0],
            "grid_dihedral_deg": base_key[1],
            **{
                f"mode_{mode}_coordinate_angstrom": coordinate
                for mode, coordinate in zip(SELECTED_MODES, mode_coordinates)
            },
        }
        if not any(mode_coordinates) and not np.array_equal(atoms.positions, base_frame["positions"]):
            raise AssertionError("Zero normal-mode coordinates must reproduce the base geometry")
        augmented_atoms.append(atoms)

expected_count = len(selected_keys) * len(NORMAL_COORDINATES) ** len(SELECTED_MODES)
if len(augmented_atoms) != expected_count:
    raise AssertionError(f"Expected {expected_count} geometries, got {len(augmented_atoms)}")

write(AUGMENTED_OUTPUT_XYZ, augmented_atoms, format="extxyz")
print(f"Wrote {len(augmented_atoms)} geometry-only 4D grid points to {AUGMENTED_OUTPUT_XYZ.resolve()}")


Wrote 6944 geometry-only 4D grid points to /home/lim_yt/X-MACE-sampling/data/A01_ethene/static/A01_ethene_grid_static_CASSCF_extended.xyz
